In [ ]:
# Install the GitHub-pinned library and its bundled profile registry.
from urllib.request import urlretrieve
from runpy import run_path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as setup_dir:
    setup_file, _ = urlretrieve(
        "https://raw.githubusercontent.com/TryDotAtwo/MultiGPUBeamSearch/25ef28f4db4d57f161644dae9fa650004a58b34d/kaggle/cube555_github/setup.py",
        setup_dir + "/setup.py",
    )
    run_path(setup_file)


In [ ]:
import cayleypy as cp
import multigpubeamsearch as beam
from multigpubeamsearch import cube555

# Puzzle selection. Set the same start and end ID to run one puzzle.
PUZZLE_ID_START = 1020
PUZZLE_ID_END = 1020
INDICES_URL = None          # Optional CSV with puzzle IDs.
RANDOM_PUZZLE = False       # Pick one puzzle at random from the selected range.
START_STATE_OVERRIDE = None # Optional custom state; use with one puzzle.

# Search limits and result format.
BEAM_WIDTH = 1_048_576      # Frontier width, or "max" for the largest admitted width.
MAX_STEPS = 140             # Maximum search depth.
BEAM_MODE = "simple"        # Standard CayleyPy beam search; the native adapter supports this mode.
USE_CHILD_SCORES = True     # Rank each generated state with the predictor.
RETURN_PATH = True          # Include the move sequence for a found solution.
NUM_GPUS = None             # None uses every GPU visible to the notebook.

# Model files. Weight 0.8 means 80% Transformer + 20% MLP; 0 means MLP only.
MODEL_ROOT = None           # Optional folder containing the puzzle model files.
COMPETITION_ROOT = None     # Optional Kaggle competition input folder.
CHECKPOINT_PATH = "q555_f1_bell2k.pt"
RESMLP_CHECKPOINT_PATH = "q555_2k_BEST.pt"
LAYOUT_PATH = "piece_layout_555.json"
TRANSFORMER_WEIGHT = 0.8

# Collect solutions, use BFS touch, and optionally reflect known solutions.
SOLUTION_MODE = "collect"
COLLECT_EXTRA_DEPTHS = 1
MAX_COLLECTED_SOLUTIONS = 2_000
TOUCH_BFS_RADIUS = 5
TOUCH_BFS_MAX_ENTRIES = 30_000_000
REFLECT_MODE = "off"      # "off", "after_original", or "only".
REFLECT_SOURCE_CSV = None  # Optional CSV containing paths to reflect.

# Result files and publication.
OUTPUT_ROOT = "cube555_runs"
PUBLISH_RESULTS = True
AUTHOR = "trydotatwo"
KAGGLE_OWNER = "trydotatwo"
KAGGLE_SLUG = "cube555-native-2xt4-blend"
KAGGLE_VERSION = 178


# Cube555 blend with CayleyPy

Edit the settings above, then choose **Run All**. The GitHub source includes an exact 2xT4 profile for the default 1,048,576 beam. The library uses exact profile matches; if none fits the chosen GPU setup and beam, it calibrates automatically.


## 1. Load the puzzle and model

Build the Cayley graph and choose the Transformer/MLP blend. The GPU backend is selected automatically.


In [ ]:
# Load puzzle data and build the Cayley graph.
# Keeping the graph on CPU leaves GPU execution under the native backend.
puzzle_ids = cube555.select_puzzles(
    PUZZLE_ID_START,
    PUZZLE_ID_END,
    indices_url=INDICES_URL,
    random_choice=RANDOM_PUZZLE,
)
puzzle = cube555.load(
    puzzle_ids[0],
    model_root=MODEL_ROOT,
    competition_root=COMPETITION_ROOT,
)
graph = cp.CayleyGraph(puzzle.definition, device="cpu", random_seed=42)

# Build the requested Transformer/MLP blend from the selected checkpoints.
predictor = puzzle.blend(
    graph,
    transformer_weight=TRANSFORMER_WEIGHT,
    transformer_checkpoint=CHECKPOINT_PATH,
    mlp_checkpoint=RESMLP_CHECKPOINT_PATH,
    layout_path=LAYOUT_PATH,
)

# Select available GPUs; matching profiles and buffer sizes are handled internally.
beam.enable_native(beam.NativeOptions(num_gpus=NUM_GPUS))


## 2. Search and save results

The native session keeps workers and model buffers ready. Valid solutions are replayed, saved, and optionally published.


In [ ]:
# Keep native workers and model buffers alive for the whole search.
with beam.NativeSession():
    for puzzle_id in puzzle_ids:
        puzzle = cube555.load(
            puzzle_id,
            model_root=MODEL_ROOT,
            competition_root=COMPETITION_ROOT,
        )
        start_state = START_STATE_OVERRIDE or puzzle.start_state

        # CayleyPy handles the search API; the native backend handles GPU work.
        result = graph.beam_search(
            start_state=start_state,
            predictor=predictor,
            backend="native",
            beam_width=BEAM_WIDTH,
            max_steps=MAX_STEPS,
            beam_mode=BEAM_MODE,
            use_child_scores=USE_CHILD_SCORES,
            return_path=RETURN_PATH,
            solution_mode=SOLUTION_MODE,
            collect_extra_depths=COLLECT_EXTRA_DEPTHS,
            max_collected_solutions=MAX_COLLECTED_SOLUTIONS,
            bfs_radius=TOUCH_BFS_RADIUS,
            bfs_max_entries=TOUCH_BFS_MAX_ENTRIES,
            reflect_mode=REFLECT_MODE,
            reflection_source=cube555.source_solution(
                REFLECT_SOURCE_CSV, puzzle_id, graph
            ),
        )

        # Replay-check, save, and optionally publish solutions through the helper.
        saved = cube555.finish_result(
            result,
            graph,
            start_state,
            publish=PUBLISH_RESULTS,
            directory=OUTPUT_ROOT,
            puzzle_id=puzzle_id,
            predictor=predictor,
            author=AUTHOR,
            kaggle_owner=KAGGLE_OWNER,
            kaggle_slug=KAGGLE_SLUG,
            kaggle_version=KAGGLE_VERSION,
        )["saved"]
        print(
            f"Puzzle {puzzle_id}: best length {result.path_length}; "
            f"solutions {len(result.solutions)}; saved {saved}"
        )
